# Stage A v2 + Stage B v2 – Google Colab

Runs **Stage A v2** (PDF extraction via Docling) then **Stage B v2** (text chunking + table SPO triples).

**Setup:** Runtime → Change runtime type → **T4 GPU** (optional but speeds up Docling).
Then run all cells **in order**.

**You need:**
1. `Thesis_llama_colab.zip` – generated by running `python create_colab_zip.py` on your PC
2. Your PDF file(s) – the medical guideline PDFs from your `input/` folder

**Output:** Two JSON files downloaded to your PC at the end:
- `stage_b_text_chunks.json` → upload this to the Stage C notebook
- `stage_b_table_triples.json` → upload this to the Stage C notebook

## Step 1: Install dependencies

In [ ]:
!pip install -q docling pymupdf pdfplumber nltk
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('[OK] Dependencies installed!')

## Step 2: Upload project ZIP

Upload `Thesis_llama_colab.zip` (created by running `python create_colab_zip.py` on your PC).

In [ ]:
import sys
import os
import zipfile
from pathlib import Path
from google.colab import files

print('[*] Upload Thesis_llama_colab.zip...')
uploaded = files.upload()

for fn in uploaded:
    if fn.endswith('.zip'):
        with zipfile.ZipFile(fn, 'r') as z:
            z.extractall('/content')
        print(f'[OK] Extracted: {fn}')
        break

# Auto-detect project root (look for pipeline/ folder)
project_dir = '/content'
for d in os.listdir('/content'):
    p = Path('/content') / d
    if p.is_dir() and (p / 'pipeline').exists():
        project_dir = str(p)
        break

if not (Path(project_dir) / 'pipeline').exists():
    raise RuntimeError('[!] pipeline/ folder not found after extraction. Make sure you uploaded the correct ZIP.')

sys.path.insert(0, project_dir)
os.chdir(project_dir)
print(f'[OK] Project root: {project_dir}')

## Step 3: Upload your PDF file(s)

Upload all your medical guideline PDFs (e.g. `Heidenreich, 2022, AHA,ACC,HFSA guidelines.pdf`).
They will be placed in the `input/` folder.

In [ ]:
import shutil
from google.colab import files
from pathlib import Path

input_dir = Path(project_dir) / 'input'
input_dir.mkdir(parents=True, exist_ok=True)

print('[*] Upload your PDF file(s)...')
uploaded_pdfs = files.upload()

saved = []
for fname in uploaded_pdfs:
    dest = input_dir / fname
    # files.upload() saves to current dir; move to input/
    src = Path(fname)
    if src.exists():
        shutil.move(str(src), str(dest))
    else:
        with open(dest, 'wb') as f:
            f.write(uploaded_pdfs[fname])
    saved.append(dest.name)
    print(f'  [+] Saved: {dest}')

if not saved:
    raise FileNotFoundError('[!] No PDFs uploaded. Run this cell again and select your PDF file(s).')

all_pdfs = list(input_dir.glob('*.pdf'))
print(f'[OK] {len(all_pdfs)} PDF(s) ready in {input_dir}')

## Step 4: Run Stage A v2 – PDF extraction (Docling)

Extracts text and tables from each PDF page using Docling.
Skips first 3 pages (cover/TOC) and last 5 pages (references).
Output goes to `outputs/STAGE_A_v2/`.

In [ ]:
import sys
import os
sys.path.insert(0, project_dir)
os.chdir(project_dir)

import config
from pipeline.data import PDFGuidelines
from pipeline.transforms import ExtractTextV2

STAGE_A_DIR = config.stage_a_dir()  # outputs/STAGE_A_v2
STAGE_A_DIR.mkdir(parents=True, exist_ok=True)

pdf_guidelines = PDFGuidelines(pdf_dir=str(Path(project_dir) / 'input'))
print(f'[*] Found {pdf_guidelines.count()} PDF(s): {[p.name for p in pdf_guidelines.get_files()]}')

if pdf_guidelines.count() == 0:
    raise FileNotFoundError('No PDFs found in input/. Run Step 3 first.')

extractor = ExtractTextV2(
    skip_first_pages=3,
    skip_last_pages=5,
    stage_output_dir=str(STAGE_A_DIR),
)
print('[*] Running Stage A v2 (this may take several minutes per PDF)...')
raw_text = extractor.transform(pdf_guidelines)

print(f'[OK] Stage A v2 complete!')
print(f'     Pages extracted: {raw_text.count()}')
print(f'     Output: {STAGE_A_DIR}')
print(f'     Files: {[f.name for f in STAGE_A_DIR.iterdir()]}')

## Step 5: Run Stage B v2 – Text chunking + table triples

Reads Stage A output, splits text into clean chunks, converts tables to SPO triples.
v2 filters out chunks that are just table-body content already captured in the triples.
Output goes to `outputs/STAGE_B_v2/`.

In [ ]:
import sys
import os
import json
sys.path.insert(0, project_dir)
os.chdir(project_dir)

import config
from pipeline.stage_io import load_stage_a_output
from pipeline.models import ParsingRules
from pipeline.transforms import ContentPreparation

STAGE_A_DIR = config.stage_a_dir()  # outputs/STAGE_A_v2
STAGE_B_DIR = config.stage_b_dir()  # outputs/STAGE_B_v2

raw_text = load_stage_a_output(STAGE_A_DIR)
if raw_text is None:
    raise FileNotFoundError('Stage A output not found. Run Step 4 first.')

print(f'[*] Loaded Stage A output: {raw_text.count()} pages')

parsing_rules = ParsingRules()
content_prep = ContentPreparation(
    parsing_rules=parsing_rules,
    min_chars=40,
    stage_output_dir=str(STAGE_B_DIR),
    stage_b_version='v2',
)
print('[*] Running Stage B v2 (chunking + table triples)...')
result = content_prep.transform(raw_text)

text_chunks = result.text_chunks
table_triples = result.table_triples
print(f'[OK] Stage B v2 complete!')
print(f'     Text chunks: {text_chunks.count()}')
print(f'     Table triples: {len(table_triples)}')
print(f'     Output: {STAGE_B_DIR}')

## Step 6: Download Stage B outputs

Download both files to your PC. You will need to **upload them both** in the next notebook (Stage C).

In [ ]:
from google.colab import files
import config
from pathlib import Path

STAGE_B_DIR = config.stage_b_dir()
chunks_file = STAGE_B_DIR / 'stage_b_text_chunks.json'
triples_file = STAGE_B_DIR / 'stage_b_table_triples.json'

if not chunks_file.exists() or not triples_file.exists():
    raise FileNotFoundError('Stage B output files not found. Run Step 5 first.')

print('[*] Downloading stage_b_text_chunks.json...')
files.download(str(chunks_file))
print('[*] Downloading stage_b_table_triples.json...')
files.download(str(triples_file))
print('[OK] Both files downloaded!')
print('     -> Next: open COLAB_Stage_C.ipynb and upload both of these files.')